# 資料清洗



## 1. 環境設定與讀取資料




In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../src")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [2]:
import os
def path(name, folder):
    """Colab 直接上傳時檔案在根目錄;下載整個 repo 執行時在資料夾裡"""
    return name if os.path.exists(name) else os.path.join(folder, name)

# 資料讀取
client_profile = pd.read_csv(path("client_profile.csv", "data_raw"))
deposit_balance = pd.read_csv(path("deposit_balance_monthly.csv", "data_raw"))
card_transactions = pd.read_csv(path("card_transactions_monthly.csv", "data_raw"))
app_engagement = pd.read_csv(path("app_engagement_monthly.csv", "data_raw"))
product_catalog = pd.read_csv(path("product_catalog.csv", "data_raw"))
conversion_label = pd.read_csv(path("conversion_label.csv", "data_raw"))

for name, df in [
    ("client_profile", client_profile),
    ("deposit_balance_monthly", deposit_balance),
    ("card_transactions_monthly", card_transactions),
    ("app_engagement_monthly", app_engagement),
    ("product_catalog", product_catalog),
    ("conversion_label", conversion_label),
]:
    print(f"{name:28s} {df.shape}")


client_profile               (9045, 12)
deposit_balance_monthly      (107738, 3)
card_transactions_monthly    (192589, 5)
app_engagement_monthly       (49936, 4)
product_catalog              (9, 5)
conversion_label             (9000, 3)


## 2. 確認資料品質


### 2.1 `client_profile` : 客戶主檔

In [3]:
# client_profile 整體型態、缺值數量、數值範圍
# .info() 看型態、.isna().sum() 看缺值、.describe() 看數值範圍
print(client_profile.info())
print("----------------------------------------------")
print(client_profile.isna().sum())
print("----------------------------------------------")
print(client_profile.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9045 entries, 0 to 9044
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             9045 non-null   object 
 1   age                     9045 non-null   int64  
 2   has_children            9045 non-null   bool   
 3   education_level         9045 non-null   object 
 4   tenure_months           8593 non-null   float64
 5   income_tier             9045 non-null   object 
 6   income_annual_wan       8018 non-null   float64
 7   has_credit_card         9045 non-null   bool   
 8   has_deposit_account     9045 non-null   bool   
 9   has_wealth_product      9045 non-null   bool   
 10  app_click_history_flag  9045 non-null   bool   
 11  kyc_risk_rating         9045 non-null   object 
dtypes: bool(5), float64(2), int64(1), object(4)
memory usage: 538.9+ KB
None
----------------------------------------------
customer_id               

In [4]:
# 檢查 education_level 有沒有同義詞/大小寫/簡繁/多餘空白等寫法不一致
client_profile.education_level.value_counts()

,count
education_level,
大學,4737
高中以下,2098
研究所以上,1719
大學,87
大学,84
Bachelor,81
高中(含)以下,56
High School,44
高中以下,37


In [5]:
# 進一步排除空白/大小寫問題
client_profile.education_level.str.strip().value_counts()

,count
education_level,
大學,4824
高中以下,2135
研究所以上,1753
大学,84
Bachelor,81
高中(含)以下,56
High School,44
Graduate,35
碩士以上,33


In [6]:
# 檢查 kyc_risk_rating 有沒有寫法不一致
client_profile.kyc_risk_rating.value_counts()

,count
kyc_risk_rating,
保守型,3194
穩健型,3177
積極型,2264
Conservative,56
保守,52
積極型,50
保守型,49
穩健,47
穩健型,46


In [7]:
# 驗證上面 kyc_risk_rating 重複的類別
# 排除空白/大小寫問題
client_profile.kyc_risk_rating.str.strip().value_counts()

,count
kyc_risk_rating,
保守型,3243
穩健型,3223
積極型,2314
Conservative,56
保守,52
穩健,47
Moderate,45
積極,33
Aggressive,32


In [8]:
# 檢查有沒有客戶被重複建檔(同一個 customer_id 出現超過一次)
client_profile.customer_id.duplicated().sum()

np.int64(45)

In [9]:
# 抓 age 超出合理範圍的異常值,列出完整明細方便之後判斷處理方式
client_profile[~client_profile["age"].between(0, 100)]

,customer_id,age,has_children,education_level,tenure_months,income_tier,income_annual_wan,has_credit_card,has_deposit_account,has_wealth_product,app_click_history_flag,kyc_risk_rating
633,C100633,999,False,大學,55.0,<40萬,32.463920,True,True,False,False,積極型
999,C100999,-3,True,高中以下,26.0,<40萬,21.538577,True,True,False,False,保守型
1716,C101716,999,False,大學,35.0,Unknown,NaN,True,False,False,False,積極型
1771,C101771,150,True,大學,24.0,<40萬,31.868784,False,True,False,True,穩健型
1930,C101930,-3,False,大學,24.0,<40萬,38.434112,True,True,False,True,積極型
2384,C102384,-3,True,研究所以上,14.0,80-120萬,77.058670,True,False,True,False,保守型
2855,C102855,999,False,高中以下,12.0,120萬+,192.852736,True,True,False,True,積極型
3579,C103579,-3,True,大學,NaN,40-60萬,53.707088,True,False,False,False,積極型
3603,C103603,150,True,研究所以上,43.0,<40萬,29.791866,False,True,False,False,積極型
3656,C103656,150,True,大學,15.0,80-120萬,92.784730,True,False,False,True,穩健型


### 2.2 `card_transactions_monthly` : 逐月信用卡消費明細



In [10]:
# card_transactions 整體型態、缺值數量、數值範圍
print(card_transactions.info())
print("----------------------------------------------")
print(card_transactions.isna().sum())
print("----------------------------------------------")
print(card_transactions.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192589 entries, 0 to 192588
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   customer_id         192589 non-null  object
 1   transaction_month   192589 non-null  object
 2   mcc_category        192589 non-null  object
 3   transaction_amount  190356 non-null  object
 4   transaction_count   192589 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 7.3+ MB
None
----------------------------------------------
customer_id              0
transaction_month        0
mcc_category             0
transaction_amount    2233
transaction_count        0
dtype: int64
----------------------------------------------
       transaction_count
count      192589.000000
mean            1.532424
std             0.864873
min             1.000000
25%             1.000000
50%             1.000000
75%             2.000000
max             9.000000


In [11]:
# 看 transaction_amount 裡除了數字之外，有沒有混進了哪些字串
card_transactions.transaction_amount.value_counts()

,count
transaction_amount,
unknown,2257
-,2253
218.0,106
202.0,105
215.0,103
...,...
16549.0,1
13909.0,1
12022.0,1


In [12]:
card_transactions["transaction_amount"].isna().sum()

np.int64(2233)

In [13]:
# 找重複的交易列
card_transactions.value_counts()

customer_id  transaction_month  mcc_category  transaction_amount  transaction_count
C107898      2026-08            精品/百貨         2563.0              1                    2
C108381      2026-08            母嬰用品          3728.0              2                    2
C107574      2026-07            外送平台          2267.0              1                    2
C101365      2026-08            居家生活          2877.0              1                    2
C101476      2026-03            超市/量販         7579.0              2                    2
                                                                                      ..
C103013      2026-06            外送平台          3944.0              1                    1
                                母嬰用品          2933.0              1                    1
                                餐飲            3821.0              1                    1
             2026-07            居家生活          3482.0              1                    1
C103012      2026-08            母嬰用品          103.0               1                    1
Name: count, Length: 189902, dtype: int64

In [14]:
# 拿到確切的重複列筆數
card_transactions.duplicated().sum()

np.int64(454)

In [15]:
# 檢查 mcc_category 有沒有寫法不一致
card_transactions.mcc_category.value_counts()

,count
mcc_category,
母嬰用品,26636
超市/量販,22254
3C科技,22148
外送平台,20789
居家生活,17699
加油/交通,17097
其他,16025
精品/百貨,16008
餐飲,16006


In [16]:
# 進一步排除空白/大小寫問題
card_transactions.mcc_category.str.strip().value_counts()


,count
mcc_category,
母嬰用品,27198
超市/量販,22254
3C科技,22148
外送平台,20789
居家生活,17699
加油/交通,17097
其他,16025
精品/百貨,16008
餐飲,16006


In [17]:
# 檢查 transaction_month 有沒有混用 "-" 和 "/" 兩種格式
# True 代表用"/"格式,False 代表用"-"格式
card_transactions["transaction_month"].astype(str).str.contains("/").value_counts()

,count
transaction_month,
False,181034
True,11555


In [18]:
# ~ : 反向(not)，isin() 回傳 True 表示有在 client_profile 裡面找到!
# 加上 ~:找不到的那些,就是孤兒列!
orphan_mask = ~card_transactions["customer_id"].isin(client_profile["customer_id"])
orphan_rows = card_transactions[orphan_mask]

print("孤兒列數:", len(orphan_rows))
print("孤兒的 customer_id 有幾個不同的:", orphan_rows["customer_id"].nunique())
print(orphan_rows["customer_id"].unique())

孤兒列數: 57
孤兒的 customer_id 有幾個不同的: 57
['C999900' 'C999901' 'C999902' 'C999903' 'C999904' 'C999905' 'C999906'
 'C999907' 'C999908' 'C999909' 'C999910' 'C999911' 'C999912' 'C999913'
 'C999914' 'C999915' 'C999916' 'C999917' 'C999918' 'C999919' 'C999920'
 'C999921' 'C999922' 'C999923' 'C999924' 'C999925' 'C999926' 'C999927'
 'C999928' 'C999929' 'C999930' 'C999931' 'C999932' 'C999933' 'C999934'
 'C999935' 'C999936' 'C999937' 'C999938' 'C999939' 'C999940' 'C999941'
 'C999942' 'C999943' 'C999944' 'C999945' 'C999946' 'C999947' 'C999948'
 'C999949' 'C999950' 'C999951' 'C999952' 'C999953' 'C999954' 'C999955'
 'C999956']


### 2.3 `deposit_balance_monthly` : 逐月活存餘額


In [19]:
# deposit_balance 整體型態、缺值數量、數值範圍
print(deposit_balance.info())
print("----------------------------------------------")
print(deposit_balance.isna().sum())
print("----------------------------------------------")
print(deposit_balance.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107738 entries, 0 to 107737
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   customer_id       107738 non-null  object 
 1   month             107738 non-null  object 
 2   checking_balance  107738 non-null  float64
dtypes: float64(1), object(2)
memory usage: 2.5+ MB
None
----------------------------------------------
customer_id         0
month               0
checking_balance    0
dtype: int64
----------------------------------------------
       checking_balance
count      1.077380e+05
mean       6.816813e+04
std        6.284744e+04
min       -8.818670e+05
25%        2.912750e+04
50%        5.083950e+04
75%        8.563250e+04
max        1.169088e+06


In [20]:
# 抓負值餘額的精確筆數(活存帳戶正常不會是負的)
(deposit_balance["checking_balance"] < 0).sum()

np.int64(215)

In [21]:
# 檢查 month 欄位有沒有混用 "-" 和 "/" 兩種格式
deposit_balance["month"].astype(str).str.contains("/").value_counts()

,count
month,
False,102352
True,5386


In [22]:
# 找出記錄沒有 12 個月的客戶
deposit_balance.groupby('customer_id').size().value_counts()

,count
12,8820
11,98
10,82


### 2.4 `app_engagement_monthly` : 逐月 App 理財瀏覽行為



In [23]:
# app_engagement 整體型態、缺值數量、數值範圍
print(app_engagement.info())
print("----------------------------------------------")
print(app_engagement.isna().sum())
print("----------------------------------------------")
print(app_engagement.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49936 entries, 0 to 49935
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  49936 non-null  object
 1   month        49936 non-null  object
 2   tag_clicked  49936 non-null  object
 3   click_count  49936 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.5+ MB
None
----------------------------------------------
customer_id    0
month          0
tag_clicked    0
click_count    0
dtype: int64
----------------------------------------------
        click_count
count  49936.000000
mean       1.757830
std        1.057782
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       14.000000


In [24]:
# 檢查 tag_clicked 有沒有寫法不一致
app_engagement.tag_clicked.value_counts()

,count
tag_clicked,
平衡型,10737
美債,10286
高股息,9948
低碳永續,9335
科技,8978
美债,179
Tech,173
US Bond,153
科技,147


In [25]:
# 進一步排除空白問題
app_engagement.tag_clicked.str.strip().value_counts()


,count
tag_clicked,
平衡型,10737
美債,10286
高股息,9948
低碳永續,9335
科技,9125
美债,179
Tech,173
US Bond,153


### 2.5 `product_catalog` : 理財商品目錄



In [26]:
# product_catalog 整體型態、缺值數量、數值範圍
print(product_catalog.info())
print("----------------------------------------------")
print(product_catalog.isna().sum())
print("----------------------------------------------")
print(product_catalog.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_id    9 non-null      object
 1   product_name  9 non-null      object
 2   asset_class   9 non-null      object
 3   category      9 non-null      object
 4   risk_rating   9 non-null      int64 
dtypes: int64(1), object(4)
memory usage: 492.0+ bytes
None
----------------------------------------------
product_id      0
product_name    0
asset_class     0
category        0
risk_rating     0
dtype: int64
----------------------------------------------
       risk_rating
count     9.000000
mean      2.666667
std       1.000000
min       1.000000
25%       2.000000
50%       3.000000
75%       3.000000
max       4.000000


### 2.6 `conversion_label` : 傾向評分模型的目標標籤



In [27]:
# conversion_label 整體型態、缺值數量、數值範圍
print(conversion_label.info())
print("----------------------------------------------")
print(conversion_label.isna().sum())
print("----------------------------------------------")
print(conversion_label.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       9000 non-null   object
 1   converted_to_sip  9000 non-null   bool  
 2   conversion_month  580 non-null    object
dtypes: bool(1), object(2)
memory usage: 149.5+ KB
None
----------------------------------------------
customer_id            0
converted_to_sip       0
conversion_month    8420
dtype: int64
----------------------------------------------
       customer_id converted_to_sip conversion_month
count         9000             9000              580
unique        9000                2                3
top        C108999            False       2026/09/01
freq             1             8420              257


## 3. 處理型態污染與異常值



### card_transactions_monthly.transaction_amount 裡的 "N/A" / "-" / "unknown" 怎麼處理?

In [28]:
#
card_transactions.loc[card_transactions["transaction_amount"] == "unknown" , "transaction_amount" ] = np.nan

In [29]:
card_transactions.loc[card_transactions["transaction_amount"] == "-" , "transaction_amount" ] = np.nan

In [30]:
card_transactions.transaction_amount.value_counts()

,count
transaction_amount,
218.0,106
202.0,105
215.0,103
196.0,99
172.0,99
...,...
13189.0,1
16549.0,1
13909.0,1


In [31]:
card_transactions["transaction_amount"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 192589 entries, 0 to 192588
Series name: transaction_amount
Non-Null Count   Dtype 
--------------   ----- 
185846 non-null  object
dtypes: object(1)
memory usage: 1.5+ MB


In [32]:
card_transactions["transaction_amount"] = pd.to_numeric(card_transactions["transaction_amount"])

In [33]:
card_transactions["transaction_amount"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 192589 entries, 0 to 192588
Series name: transaction_amount
Non-Null Count   Dtype  
--------------   -----  
185846 non-null  float64
dtypes: float64(1)
memory usage: 1.5 MB


### client_profile.age 出現的 0 / 負數 / 150 / 999 怎麼判斷、怎麼處理?
---
### `age` 異常值的比對記錄

確認資料品質時用 `client_profile[~client_profile["age"].between(0, 100)]` 抓到 26 筆異常年齡,
但 `between()` 是包含邊界值的,所以 age=0 的資料沒有被這個條件抓到。

這裡分別對 0、-3(負數)、150、999 四種可能的異常值做交叉驗證:
- age == 0: 10 筆
- age == 150: 9 筆
- age == 999: 11 筆
- age < 0: 6 筆(皆為 -3)

四種加總 = 36 筆,剛好等於 26(確認資料品質時的結果)+ 10(被 `between(0, 100)` 邊界漏掉的 age=0),
證實了確認資料品質的檢查有邊界疏漏,在這裡補齊完整清單。

In [34]:
# client_profile.age 年齡異常值處理

In [35]:
(client_profile.age == 0).sum()

np.int64(10)

In [36]:
(client_profile.age == 150).sum()

np.int64(9)

In [37]:
(client_profile.age == 999).sum()

np.int64(11)

In [38]:
(client_profile.age < 0).sum()

np.int64(6)

In [39]:
client_profile[client_profile["age"] == 0]

,customer_id,age,has_children,education_level,tenure_months,income_tier,income_annual_wan,has_credit_card,has_deposit_account,has_wealth_product,app_click_history_flag,kyc_risk_rating
1070,C101070,0,True,大學,18.0,40-60萬,58.105147,True,True,False,True,保守型
1480,C101480,0,True,大學,198.0,<40萬,26.715949,True,True,False,False,穩健型
1624,C101624,0,True,研究所以上,32.0,<40萬,22.547705,False,True,False,True,穩健型
1783,C101783,0,False,研究所以上,139.0,<40萬,38.308816,True,False,False,True,穩健型
1817,C101817,0,False,大學,91.0,<40萬,28.224597,True,True,False,False,積極型
4566,C104566,0,True,高中以下,65.0,80-120萬,95.653649,True,True,False,False,積極型
7143,C107143,0,True,研究所以上,7.0,80-120萬,62.223069,True,True,False,False,穩健型
7643,C107643,0,True,高中以下,29.0,<40萬,25.701968,True,True,False,True,穩健型
8427,C108427,0,False,大學,16.0,40-60萬,27.019892,True,False,False,False,保守型
8518,C108518,0,True,大學,71.0,<40萬,25.273727,True,True,False,False,穩健型


In [40]:
client_profile[client_profile["age"] == 150]

,customer_id,age,has_children,education_level,tenure_months,income_tier,income_annual_wan,has_credit_card,has_deposit_account,has_wealth_product,app_click_history_flag,kyc_risk_rating
1771,C101771,150,True,大學,24.0,<40萬,31.868784,False,True,False,True,穩健型
3603,C103603,150,True,研究所以上,43.0,<40萬,29.791866,False,True,False,False,積極型
3656,C103656,150,True,大學,15.0,80-120萬,92.784730,True,False,False,True,穩健型
4529,C104529,150,False,大學,48.0,<40萬,31.294563,True,True,False,False,保守型
4724,C104724,150,False,大學,32.0,<40萬,21.044511,False,True,False,False,Aggressive
6469,C106469,150,True,研究所以上,12.0,Unknown,NaN,False,True,False,True,保守型
7067,C107067,150,True,大學,85.0,Unknown,NaN,True,False,False,True,積極型
8353,C108353,150,True,高中以下,13.0,60-80萬,62.980148,False,True,False,False,積極型
8609,C108609,150,True,大學,9.0,60-80萬,63.480847,True,False,False,False,穩健型


In [41]:
client_profile[client_profile["age"] == 999]

,customer_id,age,has_children,education_level,tenure_months,income_tier,income_annual_wan,has_credit_card,has_deposit_account,has_wealth_product,app_click_history_flag,kyc_risk_rating
633,C100633,999,False,大學,55.0,<40萬,32.463920,True,True,False,False,積極型
1716,C101716,999,False,大學,35.0,Unknown,NaN,True,False,False,False,積極型
2855,C102855,999,False,高中以下,12.0,120萬+,192.852736,True,True,False,True,積極型
4202,C104202,999,True,高中以下,29.0,Unknown,NaN,True,True,False,False,穩健型
4333,C104333,999,False,大學,9.0,<40萬,25.846402,False,True,False,True,保守型
4560,C104560,999,False,大學,48.0,80-120萬,63.265229,False,True,False,False,積極型
5091,C105091,999,True,高中以下,54.0,40-60萬,32.595652,True,True,True,False,穩健型
6127,C106127,999,False,大學,159.0,80-120萬,72.304515,True,True,False,True,穩健型
8002,C108002,999,True,大學,240.0,<40萬,32.833898,True,False,False,True,積極型
8256,C108256,999,False,高中以下,18.0,80-120萬,120.325481,False,True,False,False,積極型


In [42]:
client_profile[client_profile["age"] < 0]

,customer_id,age,has_children,education_level,tenure_months,income_tier,income_annual_wan,has_credit_card,has_deposit_account,has_wealth_product,app_click_history_flag,kyc_risk_rating
999,C100999,-3,True,高中以下,26.0,<40萬,21.538577,True,True,False,False,保守型
1930,C101930,-3,False,大學,24.0,<40萬,38.434112,True,True,False,True,積極型
2384,C102384,-3,True,研究所以上,14.0,80-120萬,77.058670,True,False,True,False,保守型
3579,C103579,-3,True,大學,NaN,40-60萬,53.707088,True,False,False,False,積極型
6957,C106957,-3,False,大學,54.0,<40萬,31.772663,True,True,False,False,穩健型
6969,C106969,-3,False,高中以下,25.0,40-60萬,49.772967,False,True,False,False,積極型


In [43]:
client_profile.loc[client_profile["age"] == 0, "age" ] = np.nan

In [44]:
(client_profile.age == 0).sum()

np.int64(0)

In [45]:
client_profile.loc[client_profile["age"] == 150, "age" ] = np.nan

In [46]:
(client_profile.age == 150).sum()

np.int64(0)

In [47]:
client_profile.loc[client_profile["age"] == 999, "age" ] = np.nan

In [48]:
(client_profile.age == 999).sum()

np.int64(0)

In [49]:
client_profile.loc[client_profile["age"] < 0, "age" ] = np.nan

In [50]:
(client_profile.age < 0).sum()

np.int64(0)

In [51]:
# 最後驗證
client_profile["age"].isna().sum()

np.int64(36)

### deposit_balance_monthly.checking_balance 出現的負值怎麼處理?

In [52]:
# 將負值欄位數值替換成nan
deposit_balance.loc[deposit_balance["checking_balance"] < 0, "checking_balance" ] = np.nan


In [53]:
# 檢查checking_balance還有沒有負值的欄位
(deposit_balance["checking_balance"] < 0).sum()

np.int64(0)

### client_profile.income_annual_wan 的離群值(999999)怎麼處理?

In [54]:
#
client_profile[client_profile["income_annual_wan"] == 999999]

,customer_id,age,has_children,education_level,tenure_months,income_tier,income_annual_wan,has_credit_card,has_deposit_account,has_wealth_product,app_click_history_flag,kyc_risk_rating
194,C100194,25.0,True,大學,15.0,<40萬,999999.0,True,False,False,False,穩健型
1910,C101910,29.0,False,研究所以上,NaN,40-60萬,999999.0,True,True,True,False,積極型
2168,C102168,29.0,False,大學,46.0,60-80萬,999999.0,True,True,False,False,積極型
3807,C103807,39.0,True,大學,59.0,80-120萬,999999.0,False,True,True,False,穩健型
4051,C104051,36.0,True,大學,54.0,<40萬,999999.0,True,False,False,False,穩健型
4238,C104238,37.0,True,高中以下,14.0,80-120萬,999999.0,True,False,False,False,穩健型
4782,C104782,28.0,True,大學,23.0,<40萬,999999.0,False,True,False,False,積極型
7364,C107364,37.0,True,大學,33.0,<40萬,999999.0,True,True,False,True,穩健型
7893,C107893,31.0,False,研究所以上,85.0,<40萬,999999.0,True,True,True,False,積極型
8006,C108006,39.0,True,研究所以上,98.0,<40萬,999999.0,True,False,False,False,積極型


In [55]:
client_profile.income_tier.value_counts()

,count
income_tier,
<40萬,3135
40-60萬,1563
80-120萬,1390
60-80萬,1285
Unknown,1027
120萬+,645


In [56]:
# 將異常值欄位數值替換成nan
client_profile.loc[client_profile["income_annual_wan"] == 999999, "income_annual_wan" ] = np.nan

In [57]:
# 確認結果
(client_profile["income_annual_wan"] == 999999).sum()

np.int64(0)

## 4. 參照完整性與重複列



In [58]:
# 孤兒列處理
Mask = card_transactions["customer_id"].isin(client_profile["customer_id"])
card_transactions = card_transactions[Mask]
print("非孤兒列數:", len(card_transactions))
print("非孤兒的 customer_id 有幾個不同的:", card_transactions["customer_id"].nunique())



非孤兒列數: 192532
非孤兒的 customer_id 有幾個不同的: 9000


In [59]:
card_transactions.duplicated().sum()

np.int64(455)

In [60]:
# 重複列去重
card_transactions = card_transactions.drop_duplicates(keep='first')

In [61]:
card_transactions.duplicated().sum()

np.int64(0)

In [62]:
print("列數:", len(card_transactions))

列數: 192077


In [63]:
# client_profile 檢查重複的客戶ID
total_rows = len(client_profile)
unique_customers = client_profile["customer_id"].nunique()
duplicated_count = client_profile["customer_id"].duplicated().sum()

print(f"總列數: {total_rows}")
print(f"不重複的customer_id數: {unique_customers}")
print(f"重複的customer_id列數: {duplicated_count}")

總列數: 9045
不重複的customer_id數: 9000
重複的customer_id列數: 45


In [64]:
# 第二步:檢查重複的45筆customer_id,是整列重複還是內容衝突
dup_customers = client_profile[client_profile.duplicated(subset="customer_id", keep=False)]
dup_customers_sorted = dup_customers.sort_values("customer_id")

print(f"涉及重複customer_id的總列數: {len(dup_customers_sorted)}")
print(f"涉及的不重複customer_id數: {dup_customers_sorted['customer_id'].nunique()}")

# 檢查這些列裡面,有多少是「整列所有欄位都完全一樣」的完全重複
full_row_dup_count = dup_customers_sorted.duplicated(keep=False).sum()
print(f"其中,整列完全相同(全欄位都重複)的列數: {full_row_dup_count}")

dup_customers_sorted.head(20)

涉及重複customer_id的總列數: 90
涉及的不重複customer_id數: 45
其中,整列完全相同(全欄位都重複)的列數: 38


,customer_id,age,has_children,education_level,tenure_months,income_tier,income_annual_wan,has_credit_card,has_deposit_account,has_wealth_product,app_click_history_flag,kyc_risk_rating
104,C100104,27.0,False,高中以下,41.0,60-80萬,104.397335,True,True,False,False,積極型
9025,C100104,27.0,False,高中以下,41.0,60-80萬,104.397335,True,True,False,False,積極型
9043,C100222,38.0,False,研究所以上,50.0,Unknown,NaN,True,True,False,True,保守型
222,C100222,39.0,False,研究所以上,50.0,Unknown,NaN,True,True,False,True,保守型
330,C100330,54.0,True,大學,15.0,120萬+,178.775344,True,True,False,False,穩健型
9011,C100330,54.0,True,大學,15.0,120萬+,178.775344,True,True,False,False,穩健型
346,C100346,31.0,False,大學,65.0,60-80萬,64.073424,True,True,False,True,積極型
9042,C100346,31.0,False,大學,65.0,60-80萬,64.073424,True,True,False,True,積極型
676,C100676,64.0,True,大學,74.0,<40萬,37.663864,True,True,False,False,積極型
9036,C100676,65.0,True,大學,74.0,<40萬,37.663864,True,True,False,False,積極型


In [65]:
# 第三步:拆成「純重複」跟「內容衝突」兩組,分開看

full_dup_mask = dup_customers_sorted.duplicated(keep=False)
pure_duplicates = dup_customers_sorted[full_dup_mask]
conflicting_duplicates = dup_customers_sorted[~full_dup_mask]

print(f"純重複(整列一樣)涉及的customer_id數: {pure_duplicates['customer_id'].nunique()}")
print(f"內容衝突(customer_id一樣,其他欄位不同)涉及的customer_id數: {conflicting_duplicates['customer_id'].nunique()}")

# 檢查2-3個customer_id看差在哪個欄位
sample_conflict_ids = conflicting_duplicates["customer_id"].unique()[:3]
for cid in sample_conflict_ids:
    print(f"\ncustomer_id: {cid}")
    print(conflicting_duplicates[conflicting_duplicates["customer_id"] == cid])

純重複(整列一樣)涉及的customer_id數: 19
內容衝突(customer_id一樣,其他欄位不同)涉及的customer_id數: 26

customer_id: C100222
     customer_id   age  has_children education_level  tenure_months income_tier  income_annual_wan  has_credit_card  \
9043     C100222  38.0         False           研究所以上           50.0     Unknown                NaN             True   
222      C100222  39.0         False           研究所以上           50.0     Unknown                NaN             True   

      has_deposit_account  has_wealth_product  app_click_history_flag kyc_risk_rating  
9043                 True               False                    True             保守型  
222                  True               False                    True             保守型  

customer_id: C100676
     customer_id   age  has_children education_level  tenure_months income_tier  income_annual_wan  has_credit_card  \
676      C100676  64.0          True              大學           74.0        <40萬          37.663864             True   
9036     C100676  65.

In [66]:
# 第四步:驗證看看所有內容衝突案例,是否都只有age欄位不一樣

conflict_ids = conflicting_duplicates["customer_id"].unique()

only_age_differs = []
other_cols_differ = []

for cid in conflict_ids:
    rows = conflicting_duplicates[conflicting_duplicates["customer_id"] == cid]
    cols = rows.columns.drop("customer_id")
    diff_cols = [col for col in cols if rows[col].nunique() > 1]
    if diff_cols == ["age"]:
        only_age_differs.append(cid)
    else:
        other_cols_differ.append((cid, diff_cols))

print(f"只有age不一樣的案例數: {len(only_age_differs)} / {len(conflict_ids)}")
if other_cols_differ:
    print(f"其他欄位也不一樣的案例: {other_cols_differ}")

只有age不一樣的案例數: 26 / 26


In [67]:
# 第五步:
# Step 1: 19組純重複(全欄位含age都一樣)——標準去重,留一筆
client_profile_dedup = client_profile.drop_duplicates(keep="first").copy()
print("純重複去除後,剩餘列數:", len(client_profile_dedup))
print("此時customer_id還剩多少重複:", client_profile_dedup["customer_id"].duplicated().sum())
# 預期: 9045 - 19 = 9026 列,剩餘重複數應該剛好是26

# Step 2: 26組內容衝突(只有age不同)——保留age較大的那一筆
client_profile_final = (
    client_profile_dedup
    .sort_values(["customer_id", "age"], ascending=[True, False])
    .drop_duplicates(subset="customer_id", keep="first")
    .reset_index(drop=True)
)

print("最終列數:", len(client_profile_final))        # 預期: 9000,跟unique_customers數一致
print("customer_id是否完全唯一:", client_profile_final["customer_id"].duplicated().sum() == 0)

# Step 3: 把去重後的結果正式寫回 client_profile,讓後面所有程式碼都用這個乾淨版本
client_profile = client_profile_final


純重複去除後,剩餘列數: 9026
此時customer_id還剩多少重複: 26
最終列數: 9000
customer_id是否完全唯一: True


### 跨表邏輯一致性檢查

| 主檔欄位 | 對照的明細表 | 一致的意思 |
| --- | --- | --- |
| `has_credit_card` | `card_transactions_monthly` | 有卡 ⇔ 有刷卡紀錄 |
| `has_deposit_account` | `deposit_balance_monthly` | 有活存 ⇔ 有餘額紀錄 |
| `app_click_history_flag` | `app_engagement_monthly` | 有瀏覽紀錄旗標 ⇔ 有點擊明細 |



In [68]:
card_ids = set(card_transactions["customer_id"])
deposit_ids = set(deposit_balance["customer_id"])
app_ids = set(app_engagement["customer_id"])

checks = {
    "has_credit_card vs 有刷卡紀錄": ("has_credit_card", card_ids),
    "has_deposit_account vs 有餘額紀錄": ("has_deposit_account", deposit_ids),
    "app_click_history_flag vs 有點擊明細": ("app_click_history_flag", app_ids),
}
for title, (col, ids) in checks.items():
    print(f"\n=== {title} ===")
    print(pd.crosstab(client_profile[col], client_profile["customer_id"].isin(ids),
                      rownames=["主檔旗標"], colnames=["明細表有紀錄"]))


=== has_credit_card vs 有刷卡紀錄 ===
明細表有紀錄  True
主檔旗標        
False   2280
True    6720

=== has_deposit_account vs 有餘額紀錄 ===
明細表有紀錄  True
主檔旗標        
False   1975
True    7025

=== app_click_history_flag vs 有點擊明細 ===
明細表有紀錄  False  True 
主檔旗標                
False    6252      0
True       47   2701


In [69]:
# 旗標為 False 的客戶,他們的明細跟其他客戶有沒有差異?
monthly_spend = card_transactions.groupby("customer_id")["transaction_amount"].sum() / 6
avg_balance = deposit_balance.groupby("customer_id")["checking_balance"].mean()

print("每月刷卡中位數(依 has_credit_card):")
print(client_profile.assign(spend=client_profile["customer_id"].map(monthly_spend))
      .groupby("has_credit_card")["spend"].median().round(0))
print("\n平均活存中位數(依 has_deposit_account):")
print(client_profile.assign(balance=client_profile["customer_id"].map(avg_balance))
      .groupby("has_deposit_account")["balance"].median().round(0))

每月刷卡中位數(依 has_credit_card):
has_credit_card
False    10433.0
True     10693.0
Name: spend, dtype: float64

平均活存中位數(依 has_deposit_account):
has_deposit_account
False    50929.0
True     51183.0
Name: balance, dtype: float64


**判斷與處理:**

- **信用卡、活存**:旗標為 False 的客戶 **100%** 都有刷卡或餘額紀錄,而且金額分布和旗標為 True 的客戶幾乎一樣,代表主檔旗標有系統性錯誤。決定**以明細表的事實為準**,新增 `has_credit_card_verified`、`has_deposit_verified`,**保留原欄位**以便追溯。
    - 影響:修正後所有客戶都同時持有信用卡與活存,無法再依產品持有狀態區分客群。
- **App**:47 位客戶旗標為 True，但明細表沒有任何點擊紀錄。明細表只涵蓋 2026/03–08,合理解讀是「以前瀏覽過,但近 6 個月沒有點擊」。決定**保留原旗標**,新增 `app_click_count_6m`(近 6 個月點擊次數),這 47 位記為 0。

In [70]:
client_profile["has_credit_card_verified"] = client_profile["customer_id"].isin(card_ids)
client_profile["has_deposit_verified"] = client_profile["customer_id"].isin(deposit_ids)

# App:保留原旗標;近 6 個月點擊次數由明細加總,沒有紀錄的記為 0
app_clicks = app_engagement.groupby("customer_id")["click_count"].sum()
client_profile["app_click_count_6m"] = client_profile["customer_id"].map(app_clicks).fillna(0).astype(int)

print("has_credit_card_verified:", client_profile["has_credit_card_verified"].value_counts().to_dict())
print("has_deposit_verified:", client_profile["has_deposit_verified"].value_counts().to_dict())
print("旗標為 True 但近 6 個月點擊為 0:",
      (client_profile["app_click_history_flag"] & client_profile["app_click_count_6m"].eq(0)).sum())  # 預期 47

has_credit_card_verified: {True: 9000}
has_deposit_verified: {True: 9000}
旗標為 True 但近 6 個月點擊為 0: 47


## 5. 資料標準化

In [71]:
# 標準化
client_profile["education_level"] = client_profile["education_level"].str.strip()
client_profile["kyc_risk_rating"] = client_profile["kyc_risk_rating"].str.strip()
card_transactions["mcc_category"] = card_transactions["mcc_category"].str.strip()
app_engagement["tag_clicked"] = app_engagement["tag_clicked"].str.strip()


In [72]:
client_profile["education_level"].value_counts()

,count
education_level,
大學,4801
高中以下,2127
研究所以上,1743
大学,84
Bachelor,81
高中(含)以下,55
High School,42
Graduate,34
碩士以上,33


In [73]:
client_profile.kyc_risk_rating.str.strip().value_counts()

,count
kyc_risk_rating,
保守型,3228
穩健型,3206
積極型,2301
Conservative,56
保守,52
穩健,47
Moderate,45
積極,33
Aggressive,32


In [74]:
card_transactions.mcc_category.str.strip().value_counts()

,count
mcc_category,
母嬰用品,27116
超市/量販,22202
3C科技,22092
外送平台,20727
居家生活,17636
加油/交通,17057
其他,15989
精品/百貨,15970
餐飲,15963


In [75]:
app_engagement.tag_clicked.str.strip().value_counts()


,count
tag_clicked,
平衡型,10737
美債,10286
高股息,9948
低碳永續,9335
科技,9125
美债,179
Tech,173
US Bond,153


In [76]:
# 建立詞語對照表
education_level_map = {
    "大学" : "大學",
    "Bachelor" : "大學",
    "高中(含)以下" : "高中以下",
    "High School" : "高中以下",
    "Graduate" : "研究所以上",
    "碩士以上" : "研究所以上"
}


kyc_risk_rating_map = {
    "Conservative" : "保守型",
    "保守" : "保守型",
    "穩健" : "穩健型",
    "Moderate" : "穩健型",
    "積極" : "積極型",
    "Aggressive" : "積極型"
}

mcc_category_map = {
    "母婴用品": "母嬰用品",
    "3c科技" : "3C科技",
    "3C 科技" : "3C科技",
    "旅游" : "旅遊"
}


tag_clicked_map = {
    "美债" : "美債",
    "US Bond" : "美債",
    "Tech" : "科技"
}

In [77]:
#
client_profile["education_level"] = client_profile["education_level"].replace(education_level_map)



In [78]:
client_profile["education_level"].value_counts()

,count
education_level,
大學,4966
高中以下,2224
研究所以上,1810


In [79]:
client_profile["kyc_risk_rating"] = client_profile["kyc_risk_rating"].replace(kyc_risk_rating_map)

In [80]:
client_profile["kyc_risk_rating"].value_counts()

,count
kyc_risk_rating,
保守型,3336
穩健型,3298
積極型,2366


In [81]:
card_transactions["mcc_category"] = card_transactions["mcc_category"].replace(mcc_category_map)

In [82]:
card_transactions["mcc_category"].value_counts()

,count
mcc_category,
母嬰用品,27690
3C科技,23045
超市/量販,22202
外送平台,20727
居家生活,17636
加油/交通,17057
其他,15989
精品/百貨,15970
餐飲,15963


In [83]:
app_engagement["tag_clicked"] = app_engagement["tag_clicked"].replace(tag_clicked_map)

In [84]:
app_engagement["tag_clicked"].value_counts()

,count
tag_clicked,
平衡型,10737
美債,10618
高股息,9948
低碳永續,9335
科技,9298


In [85]:
#處理日期
# 檢查 transaction_month 有沒有混用 "-" 和 "/" 兩種格式
# True 代表用"/"格式,False 代表用"-"格式
card_transactions["transaction_month"].astype(str).str.contains("/").value_counts()


,count
transaction_month,
False,180527
True,11550


In [86]:
# 檢查 month 欄位有沒有混用 "-" 和 "/" 兩種格式
deposit_balance["month"].astype(str).str.contains("/").value_counts()

,count
month,
False,102352
True,5386


In [87]:
#把日期格式統一調成 "-"
card_transactions["transaction_month"] = card_transactions["transaction_month"].astype(str).str.replace("/", "-")
deposit_balance["month"] = deposit_balance["month"].astype(str).str.replace("/", "-")

In [88]:
card_transactions["transaction_month"] = pd.to_datetime(card_transactions["transaction_month"])
deposit_balance["month"] = pd.to_datetime(deposit_balance["month"])

In [89]:
card_transactions["transaction_month"].dtype

dtype('<M8[ns]')

In [90]:
deposit_balance["month"].dtype

dtype('<M8[ns]')

In [91]:
# 標準化完成後,再次檢查是否有被格式不一致掩蓋的重複列
dup_after_standardize = card_transactions.duplicated().sum()
print("標準化後的重複列數:", dup_after_standardize)
if dup_after_standardize > 0:
    card_transactions = card_transactions.drop_duplicates(keep="first")
    print("去重後列數:", len(card_transactions))

標準化後的重複列數: 78
去重後列數: 191999


### 依業務鍵去重:同一客戶、同一月份、同一類別只能有一列

`card_transactions_monthly` 是**彙總表**:

每位客戶、每個月、每個消費類別一列,同一個月在同一類別的多次消費已經加總在 `transaction_amount` 與 `transaction_count` 裡。所以 `(customer_id, transaction_month, mcc_category)` 重複出現,就代表是重複列。

用業務鍵去重,並**保留有真實金額的那一筆**。


In [92]:
# 依業務鍵檢查重複
key = ["customer_id", "transaction_month", "mcc_category"]
key_dup = card_transactions[card_transactions.duplicated(subset=key, keep=False)].sort_values(key)
print("依鍵重複的組數:", card_transactions.duplicated(subset=key).sum())
print("其中一筆金額為 NaN(原為哨兵值)的組數:", key_dup.groupby(key)["transaction_amount"].apply(lambda s: s.isna().sum() == 1).sum())
print("同組的交易筆數是否都相同:", key_dup.groupby(key)["transaction_count"].nunique().eq(1).all())
key_dup.head(6)

依鍵重複的組數: 42
其中一筆金額為 NaN(原為哨兵值)的組數: 42
同組的交易筆數是否都相同: True


,customer_id,transaction_month,mcc_category,transaction_amount,transaction_count
324,C100016,2026-04-01,精品/百貨,2453.0,1
192474,C100016,2026-04-01,精品/百貨,NaN,1
7112,C100343,2026-04-01,居家生活,NaN,1
192156,C100343,2026-04-01,居家生活,492.0,1
9463,C100452,2026-04-01,旅遊,NaN,1
192103,C100452,2026-04-01,旅遊,1229.0,1


In [93]:
# 去重:有金額的排前面,保留第一筆
before = len(card_transactions)
card_transactions = (
    card_transactions
    .sort_values("transaction_amount", na_position="last")
    .drop_duplicates(subset=key, keep="first")
    .sort_values(key)
    .reset_index(drop=True)
)
print("刪除列數:", before - len(card_transactions))
print("去重後列數:", len(card_transactions))
print("依鍵重複數:", card_transactions.duplicated(subset=key).sum())  # 預期 0

刪除列數: 42
去重後列數: 191957
依鍵重複數: 0


## 6. 缺失值處理

#### 找各表的缺值





In [94]:
client_profile.isna().sum()


,0
customer_id,0
age,36
has_children,0
education_level,0
tenure_months,450
income_tier,0
income_annual_wan,1035
has_credit_card,0
has_deposit_account,0
has_wealth_product,0


In [95]:
card_transactions.isna().sum()

,0
customer_id,0
transaction_month,0
mcc_category,0
transaction_amount,6695
transaction_count,0


In [96]:
deposit_balance.isna().sum()

,0
customer_id,0
month,0
checking_balance,215


In [97]:
app_engagement.isna().sum()

,0
customer_id,0
month,0
tag_clicked,0
click_count,0


In [98]:
product_catalog.isna().sum()

,0
product_id,0
product_name,0
asset_class,0
category,0
risk_rating,0


In [99]:
conversion_label.isna().sum()

,0
customer_id,0
converted_to_sip,0
conversion_month,8420


#### 確認其餘空值狀況

In [100]:
# 確認是否有客戶缺少 1-2 個月的餘額紀錄
deposit_balance.groupby("customer_id")["month"].count().value_counts()

,count
month,
12,8820
11,98
10,82


In [101]:
# 分析客戶收入為空值的原因
client_profile.isna().sum()

,0
customer_id,0
age,36
has_children,0
education_level,0
tenure_months,450
income_tier,0
income_annual_wan,1035
has_credit_card,0
has_deposit_account,0
has_wealth_product,0


In [102]:
client_profile[client_profile["income_annual_wan"].isna()]["income_tier"].value_counts()

,count
income_tier,
Unknown,1022
<40萬,8
40-60萬,2
80-120萬,2
60-80萬,1


income_tier = unknown : 沒有揭露收入

其餘的:前面處理的離群值



#### 缺失值處理

In [103]:
# age
client_profile["age"].describe()

,age
count,8964.000000
mean,37.695002
std,10.456277
min,20.000000
25%,30.000000
50%,37.000000
75%,45.000000
max,70.000000


In [104]:
client_profile["age"].skew()

np.float64(0.24447354916541889)

輕微右偏，缺失值補中位數

In [105]:
age_m = client_profile["age"].median()
client_profile["age"] = client_profile["age"].fillna(age_m)

In [106]:
client_profile["age"].isna().sum()

np.int64(0)

In [107]:
client_profile["age"].describe()

,age
count,9000.000000
mean,37.692222
std,10.435433
min,20.000000
25%,30.000000
50%,37.000000
75%,45.000000
max,70.000000


In [108]:
# tenure_months

In [109]:
client_profile["tenure_months"].describe()

,tenure_months
count,8550.000000
mean,49.103743
std,42.908247
min,3.000000
25%,21.000000
50%,36.000000
75%,62.000000
max,240.000000


In [110]:
client_profile["tenure_months"].skew()

np.float64(2.0616630629828956)

右偏

In [111]:
# 檢查缺值組 vs 非缺值組,在 income_tier 上的分布是否相似
client_profile[client_profile["tenure_months"].isna()]["income_tier"].value_counts(normalize=True)

,proportion
income_tier,
<40萬,0.382222
40-60萬,0.180000
80-120萬,0.155556
60-80萬,0.144444
Unknown,0.086667
120萬+,0.051111


In [112]:
client_profile[client_profile["tenure_months"].notna()]["income_tier"].value_counts(normalize=True)

,proportion
income_tier,
<40萬,0.345263
40-60萬,0.172749
80-120萬,0.153099
60-80萬,0.141754
Unknown,0.114971
120萬+,0.072164


In [113]:
client_profile[client_profile["tenure_months"].isna()]["age"].describe()


,age
count,450.000000
mean,38.886667
std,10.327728
min,20.000000
25%,31.000000
50%,39.000000
75%,47.000000
max,70.000000


In [114]:
client_profile[client_profile["tenure_months"].notna()]["age"].describe()

,age
count,8550.000000
mean,37.629357
std,10.437883
min,20.000000
25%,30.000000
50%,37.000000
75%,45.000000
max,70.000000


In [115]:
client_profile[client_profile["tenure_months"].isna()]["kyc_risk_rating"].value_counts(normalize=True)


,proportion
kyc_risk_rating,
穩健型,0.360000
保守型,0.328889
積極型,0.311111


In [116]:
client_profile[client_profile["tenure_months"].notna()]["kyc_risk_rating"].value_counts(normalize=True)

,proportion
kyc_risk_rating,
保守型,0.372865
穩健型,0.366784
積極型,0.260351


In [117]:
client_profile.groupby("kyc_risk_rating")["tenure_months"].median()

,tenure_months
kyc_risk_rating,
保守型,36.0
積極型,36.0
穩健型,36.0


In [118]:
# 比較:KYC 分組中位數 vs 全體中位數(這一格只比較,不補值)
tenure_median_by_risk = client_profile.groupby("kyc_risk_rating")["tenure_months"].median()
print("KYC 分組中位數:")
print(tenure_median_by_risk)
print("全體中位數:", client_profile["tenure_months"].median())

KYC 分組中位數:
kyc_risk_rating
保守型    36.0
積極型    36.0
穩健型    36.0
Name: tenure_months, dtype: float64
全體中位數: 36.0


In [119]:
# 比較:依 income_tier 分組的 tenure_months 中位數
client_profile.groupby("income_tier")["tenure_months"].median()

,tenure_months
income_tier,
120萬+,36.0
40-60萬,35.0
60-80萬,36.0
80-120萬,36.0
<40萬,36.0
Unknown,36.0


In [120]:
# 補值:全體中位數(比較結果見上兩格,分組中位數與全體中位數相同)
tenure_m = client_profile["tenure_months"].median()
client_profile["tenure_months"] = client_profile["tenure_months"].fillna(tenure_m)
client_profile["tenure_months"].isna().sum()  # 預期 0

np.int64(0)

檢查過 `kyc_risk_rating`、`income_tier` 兩個變數的分組中位數後,發現各組數值與全體中位數相同(皆為 36),分組補值不會改變結果,所以用全體中位數來補值!

檢查過兩個變數的分組關聯後,發現分組補值並不會改變結果,所以用全體中位數來補值!


In [121]:
# income_annual_wan


In [122]:
# 處理離群值來源(用income_tier中位數回補)
income_tier_m = client_profile.groupby("income_tier")["income_annual_wan"].median()
mask = client_profile["income_annual_wan"].isna() & (client_profile["income_tier"] != "Unknown")
client_profile.loc[mask, "income_annual_wan"] = client_profile.loc[mask, "income_tier"].map(income_tier_m)

In [123]:
# 標記原本是Unknown的客戶
client_profile["income_was_unknown"] = client_profile["income_tier"].eq("Unknown").astype(int)

# 剩下的NaN用全體中位數回補
income_median_overall = client_profile["income_annual_wan"].median()
client_profile["income_annual_wan"] = client_profile["income_annual_wan"].fillna(income_median_overall)


In [124]:
# 驗證
client_profile["income_annual_wan"].isna().sum()
client_profile["income_was_unknown"].value_counts()

,count
income_was_unknown,
0,7978
1,1022


In [125]:
# transaction_amount

In [126]:
# 檢查有沒有客戶全部月份的transaction_amount都缺值
valid_counts = card_transactions.groupby("customer_id")["transaction_amount"].count()
customers_with_zero_valid = valid_counts[valid_counts == 0]
print(f"完全沒有有效消費紀錄的客戶數: {len(customers_with_zero_valid)}")

完全沒有有效消費紀錄的客戶數: 0


In [127]:
# 用個人其餘時間平均補值
customer_avg_amount = card_transactions.groupby("customer_id")["transaction_amount"].transform("mean")
card_transactions["transaction_amount"] = card_transactions["transaction_amount"].fillna(customer_avg_amount)

card_transactions["transaction_amount"].isna().sum()  # 若不是0,代表有上面抓到的zero-valid客戶,要另外處理

np.int64(0)

In [128]:
# checking_balance

In [129]:
# 檢查有沒有客戶全部月份checking_balance都缺值
valid_counts_balance = deposit_balance.groupby("customer_id")["checking_balance"].count()
customers_with_zero_valid_balance = valid_counts_balance[valid_counts_balance == 0]
print(f"完全沒有有效活存紀錄的客戶數: {len(customers_with_zero_valid_balance)}")

完全沒有有效活存紀錄的客戶數: 0


In [130]:
# 判斷哪種插值方式最適合
import numpy as np
import pandas as pd

np.random.seed(42)

# 只挑12個月都沒有缺值的客戶,因為只有這些人的真實值我們全部都知道
valid_counts = deposit_balance.groupby("customer_id")["checking_balance"].apply(lambda s: s.notna().sum())
complete_customers = valid_counts[valid_counts == 12].index

df_complete = deposit_balance[deposit_balance["customer_id"].isin(complete_customers)].sort_values(["customer_id", "month"])

results = {"個人中位數": [], "個人平均數": [], "前後向補值": [], "線性插值": []}

for cust_id, g in df_complete.groupby("customer_id"):
    g = g.reset_index(drop=True)
    n = len(g)

    # 對這個客戶,12個月輪流假裝遮住一個月(而不是只測一次,結果比較穩定)
    for hide_idx in range(n):
        true_val = g.loc[hide_idx, "checking_balance"]

        masked = g["checking_balance"].copy()
        masked.iloc[hide_idx] = np.nan
        other_values = masked.dropna()

        pred_median = other_values.median()
        pred_mean = other_values.mean()
        pred_ffill = masked.ffill().bfill().iloc[hide_idx]
        pred_interp = masked.interpolate(limit_direction="both").iloc[hide_idx]

        results["個人中位數"].append(abs(pred_median - true_val))
        results["個人平均數"].append(abs(pred_mean - true_val))
        results["前後向補值"].append(abs(pred_ffill - true_val))
        results["線性插值"].append(abs(pred_interp - true_val))

for method, errors in results.items():
    print(f"{method}:平均絕對誤差 = {np.mean(errors):.2f},誤差中位數 = {np.median(errors):.2f}")

個人中位數:平均絕對誤差 = 4958.64,誤差中位數 = 2680.00
個人平均數:平均絕對誤差 = 4789.75,誤差中位數 = 2576.59
前後向補值:平均絕對誤差 = 4031.79,誤差中位數 = 2288.00
線性插值:平均絕對誤差 = 3465.10,誤差中位數 = 1959.50


線性插值誤差最小，用線性插值來補值!

In [131]:
missing_mask = deposit_balance["checking_balance"].isna()

In [132]:
# 用線性插植補值
deposit_balance["checking_balance"] = (
    deposit_balance.groupby("customer_id")["checking_balance"]
    .transform(lambda s: s.interpolate(limit_direction="both"))
)

deposit_balance = deposit_balance.sort_values(["customer_id", "month"])

In [133]:
# 確認補值後還有沒有缺值
deposit_balance["checking_balance"].isna().sum()  # 若不是0,代表有上面抓到的zero-valid客戶

np.int64(0)

In [134]:
# 檢查目前的順序有沒有照 customer_id, month 排好

is_sorted = deposit_balance.equals(
    deposit_balance.sort_values(["customer_id", "month"]).reset_index(drop=True)
)
print("資料已經照 customer_id, month 排序:", is_sorted)

資料已經照 customer_id, month 排序: True


In [135]:
# 檢查補值結果是否合理

imputed_customers = deposit_balance.loc[missing_mask, "customer_id"].unique()

sample_ids = imputed_customers[:3]   # 挑3位看看就好

for cid in sample_ids:
    sub = deposit_balance[deposit_balance["customer_id"] == cid].copy()
    sub["原本是缺值"] = missing_mask.loc[sub.index]
    print(f"\n客戶 {cid}:")
    print(sub[["month", "checking_balance", "原本是缺值"]])


客戶 C100021:
         month  checking_balance  原本是缺值
250 2025-08-01           72399.0  False
251 2025-09-01           72143.0  False
252 2025-10-01           70629.0  False
253 2025-11-01           73792.0  False
254 2025-12-01           78585.0  False
255 2026-01-01           81271.0  False
256 2026-02-01           75777.0  False
257 2026-03-01           79293.0  False
258 2026-04-01           70928.0  False
259 2026-05-01           73126.0  False
260 2026-06-01           77204.5   True
261 2026-07-01           81283.0  False

客戶 C100025:
         month  checking_balance  原本是缺值
298 2025-08-01           24112.0  False
299 2025-09-01           25596.0  False
300 2025-10-01           27346.0  False
301 2025-11-01           26593.0  False
302 2025-12-01           27680.0  False
303 2026-01-01           26770.0  False
304 2026-02-01           24567.0  False
305 2026-03-01           25326.0  False
306 2026-04-01           25148.0  False
307 2026-05-01           23923.0  False
308 2026-06-01

In [136]:
# deposit_balance month


In [137]:
# 先抓出每個客戶實際存在的月份集合,跟完整的12個月比對,找出缺的是哪幾個月
all_months = sorted(deposit_balance["month"].unique())
customer_months = deposit_balance.groupby("customer_id")["month"].apply(set)

incomplete_customers = customer_months[customer_months.apply(len) < 12]
print(f"缺行客戶數: {len(incomplete_customers)}")

missing_months_per_customer = incomplete_customers.apply(lambda s: sorted(set(all_months) - s))

# 看缺的月份集中在哪裡(例如都是最後一個月、還是隨機散落在年中)
from collections import Counter
missing_month_counter = Counter()
for months in missing_months_per_customer:
    missing_month_counter.update(months)

print("\n各月份被缺的次數:")
for m in all_months:
    m_label = pd.Timestamp(m).strftime("%Y-%m")   # 只顯示年月,不顯示時分秒
    print(f"{m_label}: {missing_month_counter.get(m, 0)}")

缺行客戶數: 180

各月份被缺的次數:
2025-08: 22
2025-09: 21
2025-10: 19
2025-11: 24
2025-12: 22
2026-01: 25
2026-02: 20
2026-03: 21
2026-04: 16
2026-05: 19
2026-06: 30
2026-07: 23


In [138]:
# leave-one-out 測試:驗證「缺1個月」跟「缺2個連續月」兩種情況
results_1month = {"個人中位數": [], "個人平均數": [], "前後向補值": [], "線性插值": []}
results_2month = {"個人中位數": [], "個人平均數": [], "前後向補值": [], "線性插值": []}

for cust_id, g in df_complete.groupby("customer_id"):
    g = g.reset_index(drop=True)
    n = len(g)

    # 缺1個月
    for hide_idx in range(n):
        true_val = g.loc[hide_idx, "checking_balance"]
        masked = g["checking_balance"].copy()
        masked.iloc[hide_idx] = np.nan
        other_values = masked.dropna()

        results_1month["個人中位數"].append(abs(other_values.median() - true_val))
        results_1month["個人平均數"].append(abs(other_values.mean() - true_val))
        pred_ffill = masked.ffill().bfill().iloc[hide_idx]
        pred_interp = masked.interpolate(limit_direction="both").iloc[hide_idx]
        results_1month["前後向補值"].append(abs(pred_ffill - true_val))
        results_1month["線性插值"].append(abs(pred_interp - true_val))

    # 缺2個連續月份
    for hide_start in range(n - 1):
        hide_idx = [hide_start, hide_start + 1]
        true_vals = g.loc[hide_idx, "checking_balance"].values
        masked = g["checking_balance"].copy()
        masked.iloc[hide_idx] = np.nan
        other_values = masked.dropna()

        pred_ffill = masked.ffill().bfill().iloc[hide_idx].values
        pred_interp = masked.interpolate(limit_direction="both").iloc[hide_idx].values

        for i, true_val in enumerate(true_vals):
            results_2month["個人中位數"].append(abs(other_values.median() - true_val))
            results_2month["個人平均數"].append(abs(other_values.mean() - true_val))
            results_2month["前後向補值"].append(abs(pred_ffill[i] - true_val))
            results_2month["線性插值"].append(abs(pred_interp[i] - true_val))

print("=== 缺1個月 ===")
for method, errors in results_1month.items():
    print(f"{method}:平均絕對誤差 = {np.mean(errors):.2f}")

print("\n=== 缺2個連續月 ===")
for method, errors in results_2month.items():
    print(f"{method}:平均絕對誤差 = {np.mean(errors):.2f}")

=== 缺1個月 ===
個人中位數:平均絕對誤差 = 4958.64
個人平均數:平均絕對誤差 = 4789.75
前後向補值:平均絕對誤差 = 4031.79
線性插值:平均絕對誤差 = 3465.10

=== 缺2個連續月 ===
個人中位數:平均絕對誤差 = 5081.94
個人平均數:平均絕對誤差 = 4837.12
前後向補值:平均絕對誤差 = 4241.84
線性插值:平均絕對誤差 = 3565.75


In [139]:
# 第一步:補出180位客戶缺的月份列

all_customers = deposit_balance["customer_id"].unique()
all_months = sorted(deposit_balance["month"].unique())

full_index = pd.MultiIndex.from_product([all_customers, all_months], names=["customer_id", "month"])

deposit_balance_full = (
    deposit_balance
    .set_index(["customer_id", "month"])
    .reindex(full_index)
    .reset_index()
)

print(f"原本列數: {len(deposit_balance)}")
print(f"補列後列數: {len(deposit_balance_full)}")
print(f"新增的列數: {len(deposit_balance_full) - len(deposit_balance)}")

原本列數: 107738
補列後列數: 108000
新增的列數: 262


In [140]:
# 第二步:幫新增的262列套用線性插值

new_row_mask = deposit_balance_full["checking_balance"].isna()

deposit_balance_full = deposit_balance_full.sort_values(["customer_id", "month"]).reset_index(drop=True)

deposit_balance_full["checking_balance"] = (
    deposit_balance_full.groupby("customer_id")["checking_balance"]
    .transform(lambda s: s.interpolate(limit_direction="both"))
)

print("補列後缺值數:", deposit_balance_full["checking_balance"].isna().sum())

補列後缺值數: 0


In [141]:
# 第三步:抽樣檢查補列的結果
# 用new_row_mask標記出哪些列是補出來的,抽3位客戶看補值是否合理

new_row_customers = deposit_balance_full.loc[new_row_mask, "customer_id"].unique()

sample_ids = new_row_customers[:3]

for cid in sample_ids:
    sub = deposit_balance_full[deposit_balance_full["customer_id"] == cid].copy()
    sub["是否為新增補列"] = new_row_mask.loc[sub.index]
    print(f"\n客戶 {cid}:")
    print(sub[["month", "checking_balance", "是否為新增補列"]])


客戶 C100018:
         month  checking_balance  是否為新增補列
216 2025-08-01      90406.000000    False
217 2025-09-01      86195.000000    False
218 2025-10-01      98987.000000    False
219 2025-11-01      92996.000000    False
220 2025-12-01      97028.000000    False
221 2026-01-01      95184.000000    False
222 2026-02-01      96160.000000    False
223 2026-03-01      97450.000000    False
224 2026-04-01     100137.666667     True
225 2026-05-01     102825.333333     True
226 2026-06-01     105513.000000    False
227 2026-07-01      96409.000000    False

客戶 C100055:
         month  checking_balance  是否為新增補列
660 2025-08-01            9112.0    False
661 2025-09-01            9306.5     True
662 2025-10-01            9501.0    False
663 2025-11-01           10498.0    False
664 2025-12-01           11234.0    False
665 2026-01-01           10748.0    False
666 2026-02-01           10650.0    False
667 2026-03-01            9662.0    False
668 2026-04-01           11680.0    False
669 2026

In [142]:
deposit_balance = deposit_balance_full.copy()

print(f"deposit_balance 列數: {len(deposit_balance)}")
print(f"缺值數: {deposit_balance['checking_balance'].isna().sum()}")

deposit_balance 列數: 108000
缺值數: 0


In [143]:
# 最後檢查

tables = {
    "client_profile": client_profile,
    "deposit_balance": deposit_balance_full,
    "card_transactions": card_transactions,
    "app_engagement": app_engagement,
    "conversion_label": conversion_label,
}

# conversion_month 只有轉化客戶才有值(converted_to_sip=False 的客戶本來就是空值),屬於預期的缺值,不列入檢查
expected_na = {"conversion_label": ["conversion_month"]}

for name, df in tables.items():
    na_counts = df.drop(columns=expected_na.get(name, [])).isna().sum()
    na_counts = na_counts[na_counts > 0]
    if na_counts.empty:
        print(f"{name}: 無缺值 ✅")
    else:
        print(f"{name}: 還有缺值 ⚠️")
        print(na_counts)

# 確認 conversion_month 的空值剛好對應未轉化客戶
print("conversion_month 空值 = 未轉化客戶數:",
      conversion_label["conversion_month"].isna().sum() == (~conversion_label["converted_to_sip"]).sum())

client_profile: 無缺值 ✅
deposit_balance: 無缺值 ✅
card_transactions: 無缺值 ✅
app_engagement: 無缺值 ✅
conversion_label: 無缺值 ✅
conversion_month 空值 = 未轉化客戶數: True


## 7. 清洗決策紀錄表

## Section 7: 清理決策紀錄表


<table style="font-size:24px; border-collapse:collapse; width:100%;">
<thead>
<tr style="background-color:#f0f0f0;">
<th style="border:1px solid #999; padding:10px; text-align:left;">問題類型</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">欄位</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">問題描述</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">處理方法</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">理由</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">驗證方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px;">型態污染/異常值</td>
<td style="border:1px solid #999; padding:10px;"><code>checking_balance</code></td>
<td style="border:1px solid #999; padding:10px;">出現負值</td>
<td style="border:1px solid #999; padding:10px;">轉NaN</td>
<td style="border:1px solid #999; padding:10px;">活存餘額不可能為負,屬無效值</td>
<td style="border:1px solid #999; padding:10px;">Section 3</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">型態污染/異常值</td>
<td style="border:1px solid #999; padding:10px;"><code>income_annual_wan</code></td>
<td style="border:1px solid #999; padding:10px;">離群值(如999999)</td>
<td style="border:1px solid #999; padding:10px;">轉NaN</td>
<td style="border:1px solid #999; padding:10px;">查證過與<code>income_tier</code>關聯性,非真實收入</td>
<td style="border:1px solid #999; padding:10px;">Section 3</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">型態污染/異常值</td>
<td style="border:1px solid #999; padding:10px;"><code>transaction_amount</code></td>
<td style="border:1px solid #999; padding:10px;">混入哨兵字串:<code>unknown</code> 2,257、<code>-</code> 2,253、<code>N/A</code> 2,233</td>
<td style="border:1px solid #999; padding:10px;"><code>unknown</code>、<code>-</code> 手動轉NaN;<code>N/A</code> 在讀檔時已由 <code>read_csv()</code> 預設轉為NaN</td>
<td style="border:1px solid #999; padding:10px;">數值欄位混入非數值字串;<code>N/A</code> 被自動轉換,需明確記錄才不會看起來像漏處理</td>
<td style="border:1px solid #999; padding:10px;">Section 3</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">型態污染/異常值</td>
<td style="border:1px solid #999; padding:10px;"><code>age</code></td>
<td style="border:1px solid #999; padding:10px;">四種異常值(0、負數、150、999)</td>
<td style="border:1px solid #999; padding:10px;">轉NaN</td>
<td style="border:1px solid #999; padding:10px;">超出合理年齡範圍</td>
<td style="border:1px solid #999; padding:10px;">Section 3</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">參照完整性</td>
<td style="border:1px solid #999; padding:10px;"><code>card_transactions_monthly</code></td>
<td style="border:1px solid #999; padding:10px;">57筆孤兒列</td>
<td style="border:1px solid #999; padding:10px;">刪除</td>
<td style="border:1px solid #999; padding:10px;">無對應<code>customer_id</code>,無法分析</td>
<td style="border:1px solid #999; padding:10px;">Section 4</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">重複列(第一次)</td>
<td style="border:1px solid #999; padding:10px;"><code>card_transactions_monthly</code></td>
<td style="border:1px solid #999; padding:10px;">454→455 筆完全重複列。多出的 1 筆是 C100775（2026-07 精品/百貨）：兩份重複列的金額分別是 - 和 N/A，哨兵值統一轉成 NaN 後才判定為重複。</td>
<td style="border:1px solid #999; padding:10px;"><code>drop_duplicates()</code>去重</td>
<td style="border:1px solid #999; padding:10px;">型態清洗後才能正確判定,清洗前髒值差異會掩蓋真重複</td>
<td style="border:1px solid #999; padding:10px;">Section 4,類別加總校驗</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">重複列(標準化後現形)</td>
<td style="border:1px solid #999; padding:10px;"><code>card_transactions_monthly</code></td>
<td style="border:1px solid #999; padding:10px;">Section5標準化完成後,又多抓到78筆重複——原本因<code>mcc_category</code>同義詞/大小寫、<code>transaction_month</code>日期格式不一致而「看起來不一樣」,標準化後字面值趨於一致才現形</td>
<td style="border:1px solid #999; padding:10px;">標準化完成後,<strong>再做一次</strong><code>drop_duplicates()</code></td>
<td style="border:1px solid #999; padding:10px;">去重是逐欄位比對字面值,不懂語意;未標準化的欄位寫法差異會讓本該判定重複的列逃過去重檢查</td>
<td style="border:1px solid #999; padding:10px;">192,077→191,999,Restart Kernel重跑驗證</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">重複列(依業務鍵)</td>
<td style="border:1px solid #999; padding:10px;"><code>card_transactions_monthly</code></td>
<td style="border:1px solid #999; padding:10px;">42組同一客戶、同一月份、同一類別出現兩列:交易筆數相同,其中一筆金額為哨兵值(重複列與哨兵值疊在同一列),整列比對抓不到</td>
<td style="border:1px solid #999; padding:10px;">標準化後、補值前,依 <code>(customer_id, transaction_month, mcc_category)</code> 去重,保留有真實金額的一筆</td>
<td style="border:1px solid #999; padding:10px;">這張表的粒度是「客戶×月份×類別」一列,同鍵出現兩次即為重複;須在補值前處理,否則NaN被補成個人平均後無法分辨真實金額</td>
<td style="border:1px solid #999; padding:10px;">依鍵重複數42→0;191,999→191,957</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">重複列</td>
<td style="border:1px solid #999; padding:10px;"><code>client_profile.customer_id</code></td>
<td style="border:1px solid #999; padding:10px;">45筆重複(19組純重複+26組僅age不同)</td>
<td style="border:1px solid #999; padding:10px;">19組標準去重;26組保留age較大者</td>
<td style="border:1px solid #999; padding:10px;">19組同Section 4邏輯;26組沒有建檔時間戳可判斷哪筆較新,兩筆年齡只差1歲,對分析幾乎無影響,採用固定、可重現的規則(保留較大者)</td>
<td style="border:1px solid #999; padding:10px;"><code>customer_id</code>唯一性驗證(9,045→9,000),已寫回<code>client_profile</code>並經Restart Kernel重跑全流程確認</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">標準化</td>
<td style="border:1px solid #999; padding:10px;">多個類別欄位</td>
<td style="border:1px solid #999; padding:10px;">頭尾空白(結構性)</td>
<td style="border:1px solid #999; padding:10px;"><code>.str.strip()</code></td>
<td style="border:1px solid #999; padding:10px;">機械化處理,無需人工判斷</td>
<td style="border:1px solid #999; padding:10px;">類別加總=已知總列數</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">標準化</td>
<td style="border:1px solid #999; padding:10px;">多個類別欄位</td>
<td style="border:1px solid #999; padding:10px;">大小寫不一致(結構性)</td>
<td style="border:1px solid #999; padding:10px;">人工對照表統一</td>
<td style="border:1px solid #999; padding:10px;">類別數不多,集中管理比自動化lower()更一致</td>
<td style="border:1px solid #999; padding:10px;">類別加總=已知總列數</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">標準化</td>
<td style="border:1px solid #999; padding:10px;">多個類別欄位</td>
<td style="border:1px solid #999; padding:10px;">繁簡/中英同義詞(語意性)</td>
<td style="border:1px solid #999; padding:10px;">人工建對照表+<code>.replace()</code></td>
<td style="border:1px solid #999; padding:10px;">無通用語法可自動判斷,需domain判斷</td>
<td style="border:1px solid #999; padding:10px;">類別加總=已知總列數</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">標準化</td>
<td style="border:1px solid #999; padding:10px;"><code>transaction_month</code>、<code>month</code></td>
<td style="border:1px solid #999; padding:10px;">日期格式混用<code>-</code>與<code>/</code></td>
<td style="border:1px solid #999; padding:10px;">統一分隔符號後<code>pd.to_datetime()</code></td>
<td style="border:1px solid #999; padding:10px;">轉為真正日期型態才能做時間序列運算</td>
<td style="border:1px solid #999; padding:10px;"><code>.dtype</code>確認為<code>datetime64[ns]</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">缺值</td>
<td style="border:1px solid #999; padding:10px;"><code>income_annual_wan</code></td>
<td style="border:1px solid #999; padding:10px;">1,035筆缺值(1,022不揭露+13離群值殘留)</td>
<td style="border:1px solid #999; padding:10px;">Step1:用<code>income_tier</code>分組中位數回補13筆離群值殘留;Step2:加旗標<code>income_was_unknown</code>;Step3:剩餘1,022筆(Unknown)用全體中位數回補</td>
<td style="border:1px solid #999; padding:10px;">拆解兩種成因,「不揭露」保留訊號不造假,「離群值殘留」按所屬收入級距回補比全體中位數更精準</td>
<td style="border:1px solid #999; padding:10px;"><code>isna()==0</code>,<code>income_was_unknown</code> value_counts:1=1,022、0=7,978</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">缺值</td>
<td style="border:1px solid #999; padding:10px;"><code>age</code></td>
<td style="border:1px solid #999; padding:10px;">偏態0.245(接近對稱)</td>
<td style="border:1px solid #999; padding:10px;">全體中位數</td>
<td style="border:1px solid #999; padding:10px;">偏態小,中位數與平均數差異可忽略</td>
<td style="border:1px solid #999; padding:10px;"><code>.skew()</code>檢查</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">缺值</td>
<td style="border:1px solid #999; padding:10px;"><code>tenure_months</code></td>
<td style="border:1px solid #999; padding:10px;">MAR(與income_tier、kyc_risk_rating中度關聯)</td>
<td style="border:1px solid #999; padding:10px;">全體中位數</td>
<td style="border:1px solid #999; padding:10px;"><code>kyc_risk_rating</code>、<code>income_tier</code> 分組中位數皆為36,與全體中位數相同,分組補值沒有額外資訊</td>
<td style="border:1px solid #999; padding:10px;">缺值組vs非缺值組交叉驗證</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">缺值</td>
<td style="border:1px solid #999; padding:10px;"><code>transaction_amount</code></td>
<td style="border:1px solid #999; padding:10px;">隨機缺值</td>
<td style="border:1px solid #999; padding:10px;">個人平均數回補</td>
<td style="border:1px solid #999; padding:10px;">未發現時間結構訊號</td>
<td style="border:1px solid #999; padding:10px;">邊界案例(全月缺值)檢查為0筆</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">缺值</td>
<td style="border:1px solid #999; padding:10px;"><code>checking_balance</code></td>
<td style="border:1px solid #999; padding:10px;">隨機缺值</td>
<td style="border:1px solid #999; padding:10px;">線性插值<code>interpolate(limit_direction="both")</code></td>
<td style="border:1px solid #999; padding:10px;">leave-one-out驗證誤差最小(3,465.10,比個人中位數4,958.64低約30%),證實月間有時間連續性</td>
<td style="border:1px solid #999; padding:10px;">leave-one-out實證+3樣本肉眼複查</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">缺值(隱性)</td>
<td style="border:1px solid #999; padding:10px;"><code>deposit_balance</code> 180位客戶</td>
<td style="border:1px solid #999; padding:10px;">缺1-2個月紀錄(整列不存在)</td>
<td style="border:1px solid #999; padding:10px;">MultiIndex補列骨架+線性插值</td>
<td style="border:1px solid #999; padding:10px;">缺月份無結構性規律(16-30次/月,非集中於首尾);線性插值在缺1月/缺2月情境下誤差皆最小</td>
<td style="border:1px solid #999; padding:10px;">新增列數262筆與理論值精準吻合+3樣本複查(含等距內插驗證)</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">跨表一致性</td>
<td style="border:1px solid #999; padding:10px;"><code>has_credit_card</code></td>
<td style="border:1px solid #999; padding:10px;">旗標為False的2,280位客戶全部有刷卡紀錄</td>
<td style="border:1px solid #999; padding:10px;">以明細為準,新增 <code>has_credit_card_verified</code>,保留原欄位</td>
<td style="border:1px solid #999; padding:10px;">矛盾率100%,且兩組每月刷卡中位數相近(10,433 vs 10,693),屬旗標系統性錯誤;修正後無法區分卡戶與存戶</td>
<td style="border:1px solid #999; padding:10px;"><code>pd.crosstab()</code> 交叉比對+分布比較</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">跨表一致性</td>
<td style="border:1px solid #999; padding:10px;"><code>has_deposit_account</code></td>
<td style="border:1px solid #999; padding:10px;">旗標為False的1,975位客戶全部有活存餘額紀錄</td>
<td style="border:1px solid #999; padding:10px;">以明細為準,新增 <code>has_deposit_verified</code>,保留原欄位</td>
<td style="border:1px solid #999; padding:10px;">矛盾率100%,且兩組平均活存中位數相近(50,929 vs 51,183),屬旗標系統性錯誤</td>
<td style="border:1px solid #999; padding:10px;"><code>pd.crosstab()</code> 交叉比對+分布比較</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">跨表一致性</td>
<td style="border:1px solid #999; padding:10px;"><code>app_click_history_flag</code></td>
<td style="border:1px solid #999; padding:10px;">旗標為True的47位客戶在App明細表中沒有任何點擊紀錄</td>
<td style="border:1px solid #999; padding:10px;">保留原旗標,新增 <code>app_click_count_6m</code>(近6個月點擊次數),這47位記為0</td>
<td style="border:1px solid #999; padding:10px;">明細表只涵蓋2026/03–08,合理解讀為「以前瀏覽過,但近6個月沒有點擊」</td>
<td style="border:1px solid #999; padding:10px;">旗標True且點擊為0的人數=47</td>
</tr>
</tbody>
</table>


## 8. 匯出清洗後的資料


In [144]:
import os

os.makedirs("data_cleaned", exist_ok=True)

cleaned_tables = {
    "client_profile_cleaned.csv": client_profile,
    "deposit_balance_cleaned.csv": deposit_balance_full,
    "card_transactions_cleaned.csv": card_transactions,
    "app_engagement_cleaned.csv": app_engagement,
    "product_catalog_cleaned.csv": product_catalog,
    "conversion_label_cleaned.csv": conversion_label,
}

for filename, df in cleaned_tables.items():
    df.to_csv(f"data_cleaned/{filename}", index=False, encoding="utf-8-sig")
    print(f"{filename}: {df.shape[0]} 列 x {df.shape[1]} 欄, 缺值數 {df.isna().sum().sum()}")

client_profile_cleaned.csv: 9000 列 x 16 欄, 缺值數 0
deposit_balance_cleaned.csv: 108000 列 x 3 欄, 缺值數 0
card_transactions_cleaned.csv: 191957 列 x 5 欄, 缺值數 0
app_engagement_cleaned.csv: 49936 列 x 4 欄, 缺值數 0
product_catalog_cleaned.csv: 9 列 x 5 欄, 缺值數 0
conversion_label_cleaned.csv: 9000 列 x 3 欄, 缺值數 8420
